In [0]:
# Create Notebook Parameter Widgets
dbutils.widgets.removeAll()
dbutils.widgets.text("ENVIRONMENT", "", "Environment")
dbutils.widgets.text("YEAR", "2020", "Trip Year")
dbutils.widgets.text("MONTH", "01", "Trip Month")
dbutils.widgets.dropdown("RESET_SINK", "False", ["True", "False"], "Reset Sink Location?")

In [0]:
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT")
YEAR = dbutils.widgets.get("YEAR")
MONTH = dbutils.widgets.get("MONTH")
RESET_SINK = (dbutils.widgets.get("RESET_SINK") == "True")

In [0]:
from pyspark.sql.functions import *

In [0]:
SOURCE         = "abfss://{CONTAINER}@wbcdaqpseudodlsa.dfs.core.windows.net/{ENVIRONMENT}"

CATALOG_NAME   = 'raw_edl_qa_poc'
SCHEMA_NAME    = ENVIRONMENT

SOURCE_FILE_NAME = f"AGG_MONTHLY_ZONE_SUMMARY_{YEAR}_{MONTH}.parquet"
TARGET_FILE_NAME = f"AGG_MONTHLY_ZONE_SUMMARY_{YEAR}_{MONTH}.csv"

SOURCE_PATH   = SOURCE.format(CONTAINER='outbound', ENVIRONMENT=ENVIRONMENT)
SOURCE_PATH_FILE = SOURCE_PATH + "/EXPORT/" + SOURCE_FILE_NAME
TARGET_PATH   = SOURCE_PATH + '/DATA_FEED/'

VOLUME_URL     = f'/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/data_feed'

TARGET_TMP       = f"{VOLUME_URL}/TMP"
TARGET           = f"{VOLUME_URL}/{TARGET_FILE_NAME}"

In [0]:
if RESET_SINK:
    spark.sql(f"DROP VOLUME IF EXISTS {VOLUME_URL}")

In [0]:
# Create data feed Location
dbutils.fs.mkdirs(TARGET_PATH)

# Create Volume
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.data_feed
  LOCATION '{TARGET_PATH}'
  COMMENT 'Data Feed Folder for {ENVIRONMENT}';
""")

In [0]:
df = spark.read.parquet(SOURCE_PATH_FILE)


In [0]:
dbutils.fs.rm(TARGET_TMP, True)
df.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(TARGET_TMP)
files = dbutils.fs.ls(TARGET_TMP)
csv_file = [f.path for f in files if f.name.endswith(".csv")][0]
dbutils.fs.mv(csv_file, TARGET)
dbutils.fs.rm(TARGET_TMP, True)